# 05.07.04 Structured Response Extraction and Python Validation

**Phase 05 — Track B: LLM-Powered Patient Interview Assistant**
**Primary Model:** `Qwen3-4B-Instruct-2507`

### Research Objectives
1. Extract demographic, household, and financial fields into structured JSON.
2. Validate extracted fields using Pydantic schemas (`PatientInterviewExtraction`) and `AssessmentValidator`.
3. Flag uncertain or missing fields for mandatory MSW confirmation.


In [1]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../../..').resolve()
PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "backend"))

from app.services.interview_assistant.extraction.structured_extraction import StructuredInterviewExtractor
from app.services.interview_assistant.validation.assessment_validator import AssessmentValidator

print("Structured extraction & Pydantic validator ready.")


Structured extraction & Pydantic validator ready.


In [ ]:
# --- Evaluate StructuredInterviewExtractor & AssessmentValidator on Uploaded Dataset Narratives ---
import matplotlib.pyplot as plt
import pandas as pd
VIZ_DIR = PROJECT_ROOT / 'reports' / 'phase_05' / 'visualizations' / 'llm_interview_assistant'
VIZ_DIR.mkdir(parents=True, exist_ok=True)
df_clean = pd.read_csv(PROJECT_ROOT / 'data' / 'interim' / 'cleaned' / 'ofphil_cleaned_tabular.csv')
combined_text = (df_clean['interview_responses'].fillna('') + ' ' + df_clean['case_study_narrative'].fillna('')).tolist()
ext_results = [StructuredInterviewExtractor.extract_from_dialogue(t) for t in combined_text]
val_results = [AssessmentValidator.validate_extraction(r)[0] for r in ext_results]
monetary_extracted = sum(1 for r in ext_results if r.monthly_household_income is not None)
hh_extracted = sum(1 for r in ext_results if r.household_size is not None)
schema_valid = sum(val_results)
fig, ax = plt.subplots(figsize=(9, 4.8))
metrics_lbl = ['Pydantic Schema Validity', 'Monetary Amount Extracted', 'Household Count Mention Extracted']
metrics_pct = [schema_valid / len(df_clean) * 100, monetary_extracted / len(df_clean) * 100, hh_extracted / len(df_clean) * 100]
bars = ax.bar(metrics_lbl, metrics_pct, color=['#1b9e77', '#2b5c8f', '#7570b3'], width=0.45, edgecolor='black')
ax.set_ylim(0, 115)
ax.set_ylabel('Percentage of Adjudicated Records (N=1,044)', fontsize=10, fontweight='bold')
ax.set_title('Structured Interview Extraction & Validation Audit on Dataset Narratives', fontsize=11.5, fontweight='bold')
for b in bars:
    h = b.get_height()
    ax.annotate(f'{h:.1f}%', xy=(b.get_x()+b.get_width()/2, h), xytext=(0, 5), textcoords='offset points', ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(VIZ_DIR / 'structured_extraction_validation_rates.png', dpi=300, bbox_inches='tight')
fig.savefig(VIZ_DIR / 'structured_extraction_validation_rates.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)
